# Forward-Forward vs Backprop under Loss of Plasticity — v3
**Eksperimen untuk naskah jurnal *Neural Networks* (Elsevier) — arketipe B (RQ-driven).**

| RQ | Pertanyaan | Diukur dengan |
|---|---|---|
| RQ1 | Apakah aturan belajar lokal (FF) kehilangan plastisitas lebih lambat daripada BP? | akurasi akhir tiap tugas, *retention* = akurasi 10 tugas terakhir / puncak |
| RQ2 | Mekanisme apa yang menjelaskannya? | unit dorman, *effective rank* representasi, besar bobot, separasi *goodness* |
| RQ3 | Apakah reset unit (ala continual backprop) saling melengkapi dengan FF? | FF+reset vs FF, BP+reset vs BP |

**Enam metode** (arsitektur MLP sama): `bp` (softmax + backprop), `bp_cbp` (bp + reset unit), `bpn` (bp + normalisasi RMS antar-layer), `bpg` (objektif *goodness* FF tetapi gradien **global**), `ff` (Forward-Forward, gradien **lokal**), `ff_cbp` (ff + reset unit).
Perbandingan kunci: **bpg vs ff** memisahkan efek *lokalitas*; **bp vs bpg** memisahkan efek *objektif*.

**Benchmark v2 (rezim memorisasi):** random-label MNIST (`rlmnist`) dan random-label CIFAR-10 (`rlcifar`): tiap tugas berisi ±1.000 gambar dengan label acak baru, dilatih banyak epoch. Plastisitas = kemampuan mencocokkan label baru (akurasi pada data tugas itu). Rezim ini dikenal cepat merusak plastisitas BP (Lyle et al., 2023).

**Baru di v3 — kontrol BP + RMS norm (`bpn`).** Di v2, BP-G mempertahankan plastisitas (*retention* ≈ 0,99) sedangkan BP runtuh. Tetapi BP dan BP-G berbeda dalam tiga hal: objektif *goodness*, label sebagai input, dan normalisasi RMS antar-layer. `bpn` = BP biasa + normalisasi RMS yang sama, untuk menguji apakah normalisasi saja sudah menjelaskan perbedaan itu. v3 juga menambah seed sampai 5 bila waktu cukup.

**Melanjutkan dari v2:** semua file v2 (`pilot.json`, `plan.json`, `lr_choice.json`, run yang `.done`) dipakai ulang; yang dijalankan hanya: tuning LR untuk `bpn`, run `bpn`, run seed yang belum selesai, lalu seed tambahan.

**Kenapa v2?** Run v1 (Permuted MNIST & label-permuted CIFAR-10, folder `results/`) menunjukkan *retention* ≈ 0,98–0,99 untuk semua metode: plastisitas belum hilang, sehingga RQ1 tidak bisa diuji. Hasil v1 tetap dipakai sebagai **rezim kontrol** dan dianalisis ulang di sel 10.

**Sel pilot (6a)** menjalankan BP saja dan menaikkan beban memorisasi bertahap sampai *retention* BP < 0,9. Grid lengkap baru dijalankan setelah rezim yang tepat ditemukan.

**Probe representasi diperbaiki:** FF/BP-G kini diukur dengan **label benar** (utama) dan label netral (pembanding), karena kode netral dapat membungkam unit yang selektif-label dan membesar-besarkan angka unit dorman.

### Batas waktu: total ≤ 2 jam
| Tahap | Sel | Perkiraan |
|---|---|---|
| Setup + unduh data | 1–5 | ±3 menit |
| Pilot BP: cari rezim yang merusak plastisitas | 6a | ±3–6 menit |
| Kalibrasi kecepatan GPU + rencana | 6 | ±2 menit |
| Tuning LR (3 nilai × 5 tugas) | 7 | ±3 menit |
| Eksperimen utama (dijadwalkan otomatis agar muat) | 8 | sisa waktu |
| Ablasi lebar (hanya jika waktu tersisa) | 9 | opsional |
| Analisis | 10 | ±2 menit |

Sel 6 mengukur detik per tugas di GPU Anda, lalu memilih jumlah **seed (5→4→3)** dan **jumlah tugas** agar eksperimen utama muat dalam anggaran. Setiap run juga berhenti dengan aman (checkpoint tersimpan) saat batas waktu tercapai.

### Cara pakai
1. `Runtime → Change runtime type → T4 GPU` (gratis). Biarkan tab browser tetap terbuka.
2. Jalankan sel 1–10 berurutan (`Runtime → Run all` juga bisa).
3. **Jika sesi terputus:** jalankan ulang semua sel. Rencana (`plan.json`) dan LR (`lr_choice.json`) dibaca ulang, dan run **melanjutkan otomatis** dari checkpoint terakhir di Google Drive; run yang selesai dilewati.
4. Sel 10 menghasilkan tabel CSV + gambar PDF/PNG 300 dpi di Drive.

Semua hasil tersimpan di `MyDrive/ff_plasticity/`. Data disimpan sebagai `uint8` agar hemat RAM.

## 1. GPU, Google Drive, dan folder kerja

In [ ]:
import os, sys, gc, json, time
SESSION_START = time.time()          # titik nol anggaran waktu sesi ini
import torch
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ff_plasticity'
DATA, RES, TUNE, FIG, CAL = (f'{BASE}/{d}' for d in ('data', 'results', 'tune', 'figures', 'calib'))
for d in (BASE, DATA, RES, TUNE, FIG, CAL):
    os.makedirs(d, exist_ok=True)

DRIVE_ROOT = '/content/drive'
def assert_drive():
    """Hentikan sebelum menulis jika Drive terlepas (tanpa ini file bisa diam-diam tersimpan di disk lokal Colab yang hilang saat sesi berakhir)."""
    if BASE.startswith(DRIVE_ROOT) and not os.path.ismount(DRIVE_ROOT):
        raise RuntimeError('Google Drive terlepas! Jalankan ulang sel 1 (drive.mount) lalu lanjutkan; hasil sebelumnya aman di Drive.')

assert_drive()
_probe = f'{BASE}/.write_test'
with open(_probe, 'w') as f:
    f.write(str(SESSION_START)); f.flush(); os.fsync(f.fileno())
assert open(_probe).read() == str(SESSION_START), 'Gagal menulis ke Google Drive'
os.remove(_probe)
import shutil
_free = shutil.disk_usage(BASE).free / 1e9
print(f'Google Drive OK: dapat ditulis di {BASE} | ruang kosong ±{_free:.1f} GB (butuh ±0.5 GB)')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '(CPU — pilih runtime GPU!)')
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv 2>/dev/null || true
!free -h | head -2

## 2. Pustaka eksperimen (ditulis ke Drive agar versinya tersimpan bersama hasil)

In [ ]:
%%writefile /content/drive/MyDrive/ff_plasticity/ffplast.py
# ffplast.py — Forward-Forward vs Backprop under loss of plasticity
# Experiment library for the Colab notebook. Every run is resumable:
# per-task metrics are appended to <run>.jsonl, a checkpoint is written
# every few tasks, and a <run>.done marker skips finished runs.

import os, json, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

METHODS = ("bp", "bp_cbp", "bpn", "bpg", "ff", "ff_cbp")
# bp      : standard MLP + softmax head, end-to-end backprop
# bp_cbp  : bp + simplified utility-based continual backprop (unit reset)
# bpg     : same architecture/objective as FF (goodness), but GLOBAL gradients
#           -> isolates the effect of locality (bpg vs ff) from the objective (bp vs bpg)
# ff      : Forward-Forward, layer-local goodness objective (Hinton, 2022)
# ff_cbp  : ff + the same unit-reset mechanism
# bpn     : bp + the same RMS normalisation before every layer as bpg/ff (softmax head, no label input)
#           -> tests whether normalisation alone explains the bp vs bpg difference


# ----------------------------------------------------------------- data
def load_dataset(name, root):
    """Return (Xtr uint8 [N,D], ytr int64, Xte, yte) as CPU tensors, cached to `root`."""
    os.makedirs(root, exist_ok=True)
    cache = os.path.join(root, f"{name}_u8.pt")
    if os.path.exists(cache):
        return torch.load(cache)
    if name == "synthetic":  # tiny dataset for smoke tests only
        g = torch.Generator().manual_seed(0)
        protos = torch.rand(10, 64, generator=g)
        def make(n):
            y = torch.randint(0, 10, (n,), generator=g)
            x = (protos[y] + 0.3 * torch.rand(n, 64, generator=g)).clamp(0, 1)
            return (x * 255).to(torch.uint8), y
        Xtr, ytr = make(3000); Xte, yte = make(1000)
    else:
        import torchvision
        if name == "mnist":
            tr = torchvision.datasets.MNIST(root, train=True, download=True)
            te = torchvision.datasets.MNIST(root, train=False, download=True)
            Xtr, ytr = tr.data.reshape(len(tr.data), -1), tr.targets.long()
            Xte, yte = te.data.reshape(len(te.data), -1), te.targets.long()
        elif name == "cifar10":
            tr = torchvision.datasets.CIFAR10(root, train=True, download=True)
            te = torchvision.datasets.CIFAR10(root, train=False, download=True)
            Xtr = torch.from_numpy(tr.data).reshape(len(tr.data), -1)
            Xte = torch.from_numpy(te.data).reshape(len(te.data), -1)
            ytr, yte = torch.tensor(tr.targets).long(), torch.tensor(te.targets).long()
        else:
            raise ValueError(name)
    out = (Xtr.contiguous(), ytr, Xte.contiguous(), yte)
    torch.save(out, cache)
    return out


class TaskStream:
    """Deterministic task sequence. kind = 'permuted' (input permutation per task)
    or 'label_permuted' (fixed images, label permutation per task)."""

    def __init__(self, data, kind, n_train, n_test, seed, device):
        Xtr, ytr, Xte, yte = data
        self.Xtr, self.ytr = Xtr.to(device), ytr.to(device)   # kept as uint8 (RAM-friendly)
        self.Xte, self.yte = Xte.to(device), yte.to(device)
        x = Xtr.float() / 255.0
        self.mean, self.std = x.mean().item(), x.std().item()
        del x
        self.kind, self.n_train, self.n_test = kind, n_train, n_test
        self.seed, self.device = seed, device
        self.dim = Xtr.shape[1]

    def _gen(self, t):
        return torch.Generator().manual_seed(1_000_003 * self.seed + t)

    def prep(self, xu8):
        return (xu8.float() / 255.0 - self.mean) / self.std

    def task(self, t):
        g = self._gen(t)
        itr = torch.randperm(len(self.ytr), generator=g)[: self.n_train].to(self.device)
        ite = torch.randperm(len(self.yte), generator=g)[: self.n_test].to(self.device)
        xtr, ytr = self.Xtr[itr], self.ytr[itr]
        xte, yte = self.Xte[ite], self.yte[ite]
        if self.kind == "permuted":
            p = torch.randperm(self.dim, generator=g).to(self.device)
            xtr, xte = xtr[:, p], xte[:, p]
        elif self.kind == "label_permuted":
            lp = torch.randperm(10, generator=g).to(self.device)
            ytr, yte = lp[ytr], lp[yte]
        elif self.kind == "random_label":
            # memorisation regime (cf. Lyle et al., 2023): fresh random labels each task;
            # plasticity = ability to fit them, so accuracy is measured on the task's own data
            ytr = torch.randint(0, 10, (len(itr),), generator=g).to(self.device)
            xte, yte = xtr[: self.n_test], ytr[: self.n_test]
        else:
            raise ValueError(self.kind)
        return xtr, ytr, xte, yte


# ---------------------------------------------------------------- model
def rms_norm(h, eps=1e-6):
    # length normalisation between layers (Hinton, 2022); scaled to unit mean square
    return h / (h.pow(2).mean(dim=1, keepdim=True).sqrt() + eps)


class Net(nn.Module):
    def __init__(self, in_dim, width, depth, mode, n_classes=10, label_energy=0.5):
        super().__init__()
        self.mode = mode
        self.goodness_based = mode in ("bpg", "ff", "ff_cbp")
        self.local = mode in ("ff", "ff_cbp")
        self.normed = self.goodness_based or mode == "bpn"
        self.in_dim, self.n_classes = in_dim, n_classes
        self.label_scale = math.sqrt(label_energy * in_dim)
        d = in_dim + (n_classes if self.goodness_based else 0)
        self.layers = nn.ModuleList()
        for _ in range(depth):
            self.layers.append(nn.Linear(d, width)); d = width
        self.head = None if self.goodness_based else nn.Linear(width, n_classes)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu"); nn.init.zeros_(m.bias)

    def embed(self, x, labels=None):
        """Concatenate a label code to the image (goodness-based nets only).
        labels=None -> neutral code (uniform 1/C), used for representation probes."""
        B = x.shape[0]
        if labels is None:
            code = torch.full((B, self.n_classes), 1.0 / self.n_classes, device=x.device)
        else:
            code = F.one_hot(labels, self.n_classes).float()
        return torch.cat([x, self.label_scale * code], dim=1)

    def hidden(self, inp):
        hs, h = [], inp
        for layer in self.layers:
            if self.normed:
                h = rms_norm(h)
                if self.local:
                    h = h.detach()        # no gradient flows between layers -> local learning
            h = F.relu(layer(h))
            hs.append(h)
        return hs

    def outgoing_weights(self, l):
        if l + 1 < len(self.layers):
            return self.layers[l + 1].weight
        return self.head.weight if self.head is not None else None


def goodness(h):
    return h.pow(2).mean(dim=1)


def ff_loss(hs_pos, hs_neg, theta):
    loss, sep = 0.0, 0.0
    for hp, hn in zip(hs_pos, hs_neg):
        gp, gn = goodness(hp), goodness(hn)
        loss = loss + F.softplus(torch.cat([theta - gp, gn - theta])).mean()
        sep += (gp.mean() - gn.mean()).item()
    return loss, sep / len(hs_pos)


def random_wrong_labels(y, n_classes, gen):
    off = torch.randint(1, n_classes, y.shape, device=y.device, generator=gen)
    return (y + off) % n_classes


@torch.no_grad()
def predict(net, x, chunk=1000):
    preds = []
    for i in range(0, x.shape[0], chunk):
        xb = x[i:i + chunk]
        if net.goodness_based:
            scores = []
            for c in range(net.n_classes):
                lab = torch.full((xb.shape[0],), c, dtype=torch.long, device=xb.device)
                hs = net.hidden(net.embed(xb, lab))
                scores.append(sum(goodness(h) for h in hs))
            preds.append(torch.stack(scores, 1).argmax(1))
        else:
            preds.append(net.head(net.hidden(xb)[-1]).argmax(1))
    return torch.cat(preds)


# -------------------------------------------- simplified continual backprop
class UnitReset:
    """Utility-based reinitialisation of low-utility hidden units, in the spirit of
    continual backprop (Dohare et al., 2024). Utility = running mean of |activation|
    x total magnitude of outgoing weights (1 for the last goodness layer)."""

    def __init__(self, net, opt, rho=1e-4, maturity=100, decay=0.99):
        self.net, self.opt = net, opt
        self.rho, self.maturity, self.decay = rho, maturity, decay
        dev = next(net.parameters()).device
        self.util = [torch.zeros(l.out_features, device=dev) for l in net.layers]
        self.age = [torch.zeros(l.out_features, device=dev) for l in net.layers]
        self.acc = [0.0 for _ in net.layers]
        self.n_reset = 0

    def state_dict(self):
        return {"util": self.util, "age": self.age, "acc": self.acc, "n_reset": self.n_reset}

    def load_state_dict(self, s):
        self.util, self.age, self.acc, self.n_reset = s["util"], s["age"], s["acc"], s["n_reset"]

    def _zero_opt_state(self, p, idx, dim):
        st = self.opt.state.get(p, {})
        for k in ("exp_avg", "exp_avg_sq", "momentum_buffer"):
            if k in st and st[k] is not None:
                st[k].index_fill_(dim, idx, 0.0)

    @torch.no_grad()
    def step(self, hs):
        for l, h in enumerate(hs):
            w_out = self.net.outgoing_weights(l)
            out_mag = w_out.abs().sum(0) if w_out is not None else torch.ones_like(self.util[l])
            contrib = h.abs().mean(0) * out_mag
            self.util[l].mul_(self.decay).add_((1 - self.decay) * contrib)
            self.age[l] += 1
            self.acc[l] += self.rho * h.shape[1]
            n = int(self.acc[l])
            if n < 1:
                continue
            self.acc[l] -= n
            eligible = (self.age[l] > self.maturity).nonzero().flatten()
            if eligible.numel() == 0:
                continue
            k = min(n, eligible.numel())
            idx = eligible[torch.topk(self.util[l][eligible], k, largest=False).indices]
            layer = self.net.layers[l]
            fan_in = layer.weight.shape[1]
            layer.weight[idx] = torch.randn(k, fan_in, device=idx.device) * math.sqrt(2.0 / fan_in)
            layer.bias[idx] = 0.0
            self._zero_opt_state(layer.weight, idx, 0)
            self._zero_opt_state(layer.bias, idx, 0)
            if w_out is not None:
                w_out[:, idx] = 0.0
                self._zero_opt_state(w_out, idx, 1)
            self.util[l][idx] = 0.0
            self.age[l][idx] = 0.0
            self.n_reset += k


# ---------------------------------------------------------------- metrics
def _dormant_erank(hs):
    dormant, erank = [], []
    for h in hs:
        active_frac = (h > 0).float().mean(0)
        dormant.append((active_frac <= 0.01).float().mean().item())
        s = torch.linalg.svdvals(h)
        p = s / (s.sum() + 1e-12)
        p = p[p > 0]
        erank.append(float(torch.exp(-(p * p.log()).sum())))
    return dormant, erank


@torch.no_grad()
def representation_metrics(net, x_probe, y_probe=None):
    """Goodness-based nets are probed with the TRUE label code (the input they process during
    learning; primary metric) and with a neutral code (secondary). A neutral code can silence
    label-selective units and so inflate dormancy; reporting both makes that visible."""
    out = {"wmag": [l.weight.abs().mean().item() for l in net.layers]}
    if net.goodness_based:
        out["dormant"], out["erank"] = _dormant_erank(net.hidden(net.embed(x_probe, y_probe)))
        out["dormant_neutral"], out["erank_neutral"] = _dormant_erank(net.hidden(net.embed(x_probe)))
    else:
        out["dormant"], out["erank"] = _dormant_erank(net.hidden(x_probe))
        out["dormant_neutral"] = out["erank_neutral"] = None
    return out


# ------------------------------------------------------------------ runner
def run_id(bench, method, width, seed):
    return f"{bench}_{method}_w{width}_s{seed}"


def _sync():
    # push OS buffers to the Google Drive FUSE layer so uploads start immediately
    if hasattr(os, "sync"):
        os.sync()


def _atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    with open(tmp, "rb") as f:
        os.fsync(f.fileno())
    os.replace(tmp, path)
    _sync()


def _append_jsonl(path, rec):
    with open(path, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())


def _truncate_jsonl(path, keep_below_task):
    if not os.path.exists(path):
        return
    with open(path) as f:
        lines = [ln for ln in f if ln.strip() and json.loads(ln)["task"] < keep_below_task]
    with open(path, "w") as f:
        f.writelines(lines)


def make_optimizer(params, name, lr):
    if name == "adam":
        return torch.optim.Adam(params, lr=lr)
    if name == "sgd":
        return torch.optim.SGD(params, lr=lr)
    raise ValueError(name)


def run(bench_cfg, method, seed, width, depth, lr, out_dir, data, device,
        optimizer="adam", theta=2.0, label_energy=0.5, rho=1e-4, maturity=100,
        ckpt_every=10, n_tasks=None, log=print, tag=None, deadline=None):
    """Train one (benchmark, method, width, seed) run over the task sequence. Resumable.
    deadline: wall-clock time (time.time()) after which the run checkpoints and returns None."""
    os.makedirs(out_dir, exist_ok=True)
    rid = tag or run_id(bench_cfg["name"], method, width, seed)
    jsonl = os.path.join(out_dir, rid + ".jsonl")
    ckpt = os.path.join(out_dir, rid + ".ckpt")
    done = os.path.join(out_dir, rid + ".done")
    n_tasks = n_tasks or bench_cfg["n_tasks"]
    if os.path.exists(done):
        log(f"[skip] {rid} already finished"); return rid

    torch.manual_seed(seed); np.random.seed(seed)
    stream = TaskStream(data, bench_cfg["kind"], bench_cfg["n_train"], bench_cfg["n_test"], seed, device)
    net = Net(stream.dim, width, depth, method, label_energy=label_energy).to(device)
    opt = make_optimizer(net.parameters(), optimizer, lr)
    reset = UnitReset(net, opt, rho=rho, maturity=maturity) if method.endswith("_cbp") else None

    start = 0
    if os.path.exists(ckpt):
        s = torch.load(ckpt, map_location=device, weights_only=False)
        net.load_state_dict(s["net"]); opt.load_state_dict(s["opt"])
        if reset is not None: reset.load_state_dict(s["reset"])
        torch.set_rng_state(s["rng_cpu"])
        if device.type == "cuda" and s.get("rng_cuda") is not None:
            torch.cuda.set_rng_state(s["rng_cuda"])
        start = s["task"]
        log(f"[resume] {rid} from task {start}")
    _truncate_jsonl(jsonl, start)

    bs, epochs = bench_cfg["batch"], bench_cfg["epochs"]
    t0 = time.time()
    for t in range(start, n_tasks):
        xtr, ytr, xte, yte = stream.task(t)
        gdev = torch.Generator(device=device).manual_seed(7919 * seed + t)
        steps_per_epoch = max(1, xtr.shape[0] // bs)
        total = steps_per_epoch * epochs
        q_step = max(1, total // 4)
        acc_q1, sep_sum, loss_sum = None, 0.0, 0.0
        x_eval = stream.prep(xte)
        step = 0
        net.train()
        for _ in range(epochs):
            order = torch.randperm(xtr.shape[0], device=device, generator=gdev)
            for i in range(steps_per_epoch):
                b = order[i * bs:(i + 1) * bs]
                xb, yb = stream.prep(xtr[b]), ytr[b]
                if net.goodness_based:
                    yneg = random_wrong_labels(yb, net.n_classes, gdev)
                    hs_pos = net.hidden(net.embed(xb, yb))
                    hs_neg = net.hidden(net.embed(xb, yneg))
                    loss, sep = ff_loss(hs_pos, hs_neg, theta)
                    sep_sum += sep; hs_for_reset = hs_pos
                else:
                    hs = net.hidden(xb)
                    loss = F.cross_entropy(net.head(hs[-1]), yb)
                    hs_for_reset = hs
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
                loss_sum += loss.item() if (step % 20 == 0) else 0.0
                if reset is not None:
                    reset.step([h.detach() for h in hs_for_reset])
                step += 1
                if step == q_step:
                    net.eval()
                    acc_q1 = (predict(net, x_eval) == yte).float().mean().item()
                    net.train()
        net.eval()
        acc_end = (predict(net, x_eval) == yte).float().mean().item()
        npb = min(1000, x_eval.shape[0])
        rep = representation_metrics(net, x_eval[:npb], yte[:npb])
        rec = {"run": rid, "bench": bench_cfg["name"], "method": method, "width": width,
               "seed": seed, "lr": lr, "task": t, "acc_q1": acc_q1, "acc_end": acc_end,
               "goodness_sep": (sep_sum / step) if net.goodness_based else None,
               "n_reset": reset.n_reset if reset is not None else 0,
               "dormant": rep["dormant"], "erank": rep["erank"], "wmag": rep["wmag"],
               "dormant_neutral": rep["dormant_neutral"], "erank_neutral": rep["erank_neutral"],
               "elapsed_s": round(time.time() - t0, 1)}
        _append_jsonl(jsonl, rec)
        out_of_time = deadline is not None and time.time() > deadline and t + 1 < n_tasks
        if (t + 1) % ckpt_every == 0 or t + 1 == n_tasks or out_of_time:
            _atomic_save({"net": net.state_dict(), "opt": opt.state_dict(),
                          "reset": reset.state_dict() if reset is not None else None,
                          "rng_cpu": torch.get_rng_state(),
                          "rng_cuda": torch.cuda.get_rng_state() if device.type == "cuda" else None,
                          "task": t + 1}, ckpt)
        if t % 10 == 0 or t + 1 == n_tasks:
            log(f"{rid} task {t:4d} acc_end={acc_end:.3f} dormant={np.mean(rep['dormant']):.3f} "
                f"erank={np.mean(rep['erank']):.1f} ({time.time() - t0:.0f}s)")
        del xtr, ytr, xte, yte, x_eval
        if out_of_time:
            log(f"[pause] {rid} at task {t + 1}/{n_tasks}: time budget reached, checkpoint saved")
            return None
    with open(done, "w") as f:
        f.write("ok"); f.flush(); os.fsync(f.fileno())
    if os.path.exists(ckpt):
        os.remove(ckpt)   # keep Drive tidy; jsonl holds all results
    _sync()
    return rid


# ------------------------------------------------------- time budgeting
def seconds_per_task(bench_cfg, method, width, depth, lr, data, device, tmp_dir, n=3, **kw):
    """Measure wall-clock seconds per task on this GPU (first task excluded as warm-up)."""
    tag = f"cal_{bench_cfg['name']}_{method}_w{width}"
    for ext in (".jsonl", ".ckpt", ".done"):
        p = os.path.join(tmp_dir, tag + ext)
        if os.path.exists(p):
            os.remove(p)
    run(bench_cfg, method, 0, width, depth, lr, tmp_dir, data, device, n_tasks=n, tag=tag,
        log=lambda *a: None, **kw)
    with open(os.path.join(tmp_dir, tag + ".jsonl")) as f:
        e = [json.loads(ln)["elapsed_s"] for ln in f if ln.strip()]
    return (e[-1] - e[0]) / (len(e) - 1) if len(e) > 1 else e[0]


def make_plan(spt, bench, budget_s, min_tasks, seed_options=(5, 4, 3)):
    """Choose the number of seeds and tasks so the main grid fits in budget_s.
    spt[bench][method] = seconds per task. All benchmarks are scaled by the same factor,
    preferring more seeds as long as every benchmark keeps at least min_tasks[bench] tasks."""
    best = None
    for ns in seed_options:
        per_task = {b: ns * sum(spt[b].values()) for b in bench}
        full = sum(per_task[b] * bench[b]["n_tasks"] for b in bench)
        f = min(1.0, budget_s / full)
        nt = {b: int(bench[b]["n_tasks"] * f) for b in bench}
        plan = {"seeds": ns, "n_tasks": nt, "scale": round(f, 3),
                "est_min": round(sum(per_task[b] * nt[b] for b in bench) / 60, 1)}
        if all(nt[b] >= min_tasks[b] for b in bench):
            return plan
        best = plan
    nt = {b: min_tasks[b] for b in bench}       # fallback: minimum tasks, fewest seeds
    ns = seed_options[-1]
    best.update(seeds=ns, n_tasks=nt, scale=None,
                est_min=round(sum(ns * sum(spt[b].values()) * nt[b] for b in bench) / 60, 1),
                warning="budget too small for min_tasks; plan exceeds budget")
    return best


def pilot_retention(bench_cfg, data, device, out_dir, lr=1e-3, n_tasks=40, width=500, depth=3,
                    window=10, **kw):
    """Run BP alone (seed 0) to check that the regime actually removes plasticity.
    Returns (retention, acc_peak, acc_last). Deletes and reruns any earlier pilot of the same config."""
    tag = f"pilot_{bench_cfg['name']}_n{bench_cfg['n_train']}_e{bench_cfg['epochs']}"
    for ext in (".jsonl", ".ckpt", ".done"):
        p = os.path.join(out_dir, tag + ext)
        if os.path.exists(p):
            os.remove(p)
    run(bench_cfg, "bp", 0, width, depth, lr, out_dir, data, device, n_tasks=n_tasks, tag=tag,
        log=lambda *a: None, **kw)
    with open(os.path.join(out_dir, tag + ".jsonl")) as f:
        acc = np.array([json.loads(ln)["acc_end"] for ln in f if ln.strip()])
    roll = np.convolve(acc, np.ones(window) / window, mode="valid") if len(acc) >= window else acc
    peak, last = float(roll.max()), float(acc[-window:].mean())
    return last / peak, peak, last


# ---------------------------------------------------------------- analysis
def load_results(out_dir):
    import pandas as pd
    rows = []
    for fn in sorted(os.listdir(out_dir)):
        if fn.endswith(".jsonl"):
            with open(os.path.join(out_dir, fn)) as f:
                rows += [json.loads(ln) for ln in f if ln.strip()]
    df = pd.DataFrame(rows)
    if len(df):
        for k in ("dormant", "erank", "wmag", "dormant_neutral", "erank_neutral"):
            if k in df:
                df[k + "_mean"] = df[k].apply(lambda v: np.mean(v) if isinstance(v, list) else np.nan)
    return df


def plasticity_table(df, window=10, last=10):
    """Per run: peak = best rolling-`window` mean of end-of-task accuracy, acc_last = mean over
    the last `last` tasks, retention = acc_last / peak (1 = no loss of plasticity).
    Using the peak (not the first tasks) avoids rewarding methods that merely start slowly."""
    import pandas as pd
    recs = []
    for (bench, method, width, seed), g in df.groupby(["bench", "method", "width", "seed"]):
        g = g.sort_values("task")
        roll = g["acc_end"].rolling(window, min_periods=1).mean()
        peak = roll.max(); a1 = g["acc_end"].tail(last).mean()
        recs.append({"bench": bench, "method": method, "width": width, "seed": seed,
                     "acc_peak": peak, "task_peak": int(g["task"].iloc[int(roll.values.argmax())]),
                     "acc_last": a1, "retention": a1 / peak if peak > 0 else np.nan,
                     "acc_q1_last": g["acc_q1"].tail(last).mean(),
                     "dormant_last": g["dormant_mean"].tail(last).mean(),
                     "dormant_neutral_last": g["dormant_neutral_mean"].tail(last).mean() if "dormant_neutral_mean" in g else np.nan,
                     "erank_neutral_last": g["erank_neutral_mean"].tail(last).mean() if "erank_neutral_mean" in g else np.nan,
                     "erank_last": g["erank_mean"].tail(last).mean(),
                     "n_tasks": g["task"].max() + 1})
    per_run = pd.DataFrame(recs)
    summ = per_run.groupby(["bench", "method", "width"]).agg(["mean", "std", "count"])
    return per_run, summ


def pairwise_tests(per_run, metric="retention", pairs=(("ff", "bpg"), ("ff", "bp"), ("bpg", "bp"),
                                                         ("ff_cbp", "ff"), ("bp_cbp", "bp"),
                                                         ("bpn", "bp"), ("bpg", "bpn"))):
    """Welch t-test + Mann-Whitney U per benchmark/width for the method pairs."""
    from scipy import stats
    import pandas as pd
    out = []
    for (bench, width), g in per_run.groupby(["bench", "width"]):
        for a, b in pairs:
            xa = g.loc[g.method == a, metric].dropna(); xb = g.loc[g.method == b, metric].dropna()
            if len(xa) < 2 or len(xb) < 2:
                continue
            t = stats.ttest_ind(xa, xb, equal_var=False)
            u = stats.mannwhitneyu(xa, xb, alternative="two-sided")
            d = (xa.mean() - xb.mean()) / math.sqrt((xa.var(ddof=1) + xb.var(ddof=1)) / 2 + 1e-12)
            out.append({"bench": bench, "width": width, "A": a, "B": b, "metric": metric,
                        "mean_A": xa.mean(), "mean_B": xb.mean(), "cohen_d": d,
                        "welch_p": t.pvalue, "mwu_p": u.pvalue, "n_A": len(xa), "n_B": len(xb)})
    return pd.DataFrame(out)


def plot_curves(df, bench, width, metric, out_path, smooth=5, ylabel=None):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    labels = {"bp": "BP", "bp_cbp": "BP + reset", "bpg": "BP-G (global goodness)",
              "ff": "FF (local)", "ff_cbp": "FF + reset", "bpn": "BP + RMS norm"}
    styles = {"bp": ("#1f77b4", "-"), "bp_cbp": ("#1f77b4", "--"), "bpg": ("#7f7f7f", "-"),
              "ff": ("#d62728", "-"), "ff_cbp": ("#d62728", "--"), "bpn": ("#2ca02c", "-")}
    sub = df[(df.bench == bench) & (df.width == width)]
    fig, ax = plt.subplots(figsize=(4.2, 3.0), dpi=300)
    for m in METHODS:
        g = sub[sub.method == m]
        if g.empty:
            continue
        piv = g.pivot_table(index="task", columns="seed", values=metric)
        piv = piv.rolling(smooth, min_periods=1).mean()
        mu, sd = piv.mean(1), piv.std(1).fillna(0)
        c, ls = styles[m]
        ax.plot(mu.index, mu.values, color=c, ls=ls, lw=1.2, label=labels[m])
        ax.fill_between(mu.index, mu - sd, mu + sd, color=c, alpha=0.15, lw=0)
    ax.set_xlabel("Task number"); ax.set_ylabel(ylabel or metric)
    ax.grid(alpha=0.3, lw=0.5); ax.legend(fontsize=6, frameon=False)
    fig.tight_layout(); fig.savefig(out_path); plt.close(fig)
    return out_path


## 3. Import pustaka

In [ ]:
sys.path.insert(0, BASE)
import importlib, ffplast as fp
importlib.reload(fp)
print('Metode:', fp.METHODS)

## 4. Konfigurasi
Nilai `n_tasks` di bawah adalah **batas atas**; sel 6 dapat menurunkannya (tidak di bawah `MIN_TASKS`) agar total muat dalam `BUDGET_MIN`.
Untuk uji cepat (±5 menit), set `QUICK = True`.

In [ ]:
QUICK = False

BUDGET_MIN   = 120                   # total waktu sesi (menit)
RESERVE_MIN  = 8                     # cadangan untuk analisis + jeda I/O Drive

BENCH = {   # n_train/epochs di bawah = titik awal; sel pilot 6a dapat menaikkannya
    'rlmnist': dict(name='rlmnist', dataset='mnist',   kind='random_label',
                    n_train=1000, n_test=1000, batch=64, epochs=20, n_tasks=100),
    'rlcifar': dict(name='rlcifar', dataset='cifar10', kind='random_label',
                    n_train=1000, n_test=1000, batch=64, epochs=20, n_tasks=100),
}
MIN_TASKS    = {'rlmnist': 40, 'rlcifar': 40}
PILOT_LADDER = [dict(n_train=1000, epochs=20), dict(n_train=1000, epochs=40), dict(n_train=2000, epochs=40)]
PILOT_TASKS, PILOT_TARGET = 40, 0.90   # rezim dipakai bila retention BP < 0,90 dalam 40 tugas

RES, TUNE, CAL, FIG, PIL = (f'{BASE}/{d}_v2' for d in ('results', 'tune', 'calib', 'figures', 'pilot'))
CONTROL_RES = f'{BASE}/results'       # hasil v1 = rezim kontrol
for d in (RES, TUNE, CAL, FIG, PIL): os.makedirs(d, exist_ok=True)
SEED_OPTIONS = (5, 4, 3)             # dicoba berurutan oleh perencana
WIDTH, DEPTH = 500, 3                # MLP 3 hidden layer x 500 unit
OPTIMIZER    = 'adam'
THETA        = 2.0                   # ambang goodness FF
LABEL_ENERGY = 0.5                   # energi kode label relatif terhadap citra (FF/BP-G)
RHO, MATURITY = 1e-4, 100            # laju reset unit & umur minimal (continual backprop)
CKPT_EVERY   = 10                    # checkpoint ke Drive tiap 10 tugas
LR_GRID      = [3e-4, 1e-3, 3e-3]
TUNE_TASKS, TUNE_SEED = 5, 100       # tuning hanya pada 5 tugas awal, seed terpisah dari evaluasi
CAL_TASKS    = 3                     # tugas per metode untuk kalibrasi kecepatan

if QUICK:
    BUDGET_MIN, RESERVE_MIN = 6, 1
    for b in BENCH.values():
        b.update(n_tasks=12)
    for b in BENCH.values():
        b.update(n_train=500, n_test=500, epochs=5)
    MIN_TASKS = {'rlmnist': 6, 'rlcifar': 6}
    PILOT_LADDER, PILOT_TASKS = [dict(n_train=500, epochs=5), dict(n_train=500, epochs=10)], 8
    SEED_OPTIONS, TUNE_TASKS = (2, 1), 2
    RES, TUNE, CAL, FIG, PIL = (f'{BASE}/{d}_v2_quick' for d in ('results', 'tune', 'calib', 'figures', 'pilot'))
    for d in (RES, TUNE, CAL, FIG, PIL): os.makedirs(d, exist_ok=True)

DEADLINE = SESSION_START + (BUDGET_MIN - RESERVE_MIN) * 60
KW = dict(optimizer=OPTIMIZER, theta=THETA, label_energy=LABEL_ENERGY, rho=RHO, maturity=MATURITY)
def minutes_left():
    return (DEADLINE - time.time()) / 60
print(f'Sisa waktu untuk eksperimen: {minutes_left():.1f} menit')

## 5. Muat data (unduh sekali, di-cache sebagai uint8 di Drive)

In [ ]:
_cache = {}
def get_data(bench_key):
    """Hanya satu dataset di memori pada satu waktu (hemat RAM)."""
    name = BENCH[bench_key]['dataset']
    if name not in _cache:
        _cache.clear(); gc.collect()
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        _cache[name] = fp.load_dataset(name, DATA)
    return _cache[name]

for k in BENCH:
    Xtr, ytr, Xte, yte = get_data(k)
    print(k, 'train', tuple(Xtr.shape), 'test', tuple(Xte.shape), Xtr.dtype)

## 6a. Pilot: pastikan BP benar-benar kehilangan plastisitas
BP (seed 0, LR 1e-3) dijalankan 40 tugas per konfigurasi. Konfigurasi dinaikkan bertahap (`PILOT_LADDER`) sampai *retention* BP < 0,90. Pilihan disimpan ke `pilot.json` dan dipakai oleh semua sel berikutnya. Jika tidak ada konfigurasi yang lolos, **jangan lanjutkan grid** — laporkan hasilnya dulu.

In [ ]:
PILOT_FILE = f'{RES}/pilot.json'
if os.path.exists(PILOT_FILE):
    PILOT = json.load(open(PILOT_FILE)); print('Pilot lama dipakai:', PILOT)
else:
    PILOT = {}
    for bk, cfg in BENCH.items():
        data = get_data(bk)
        for step in PILOT_LADDER:
            c = dict(cfg, **step); c['n_test'] = min(c['n_test'], c['n_train'])
            assert_drive()
            R, peak, last = fp.pilot_retention(c, data, DEVICE, PIL, lr=1e-3, n_tasks=PILOT_TASKS,
                                               width=WIDTH, depth=DEPTH, **KW)
            print(f'{bk}: n_train={step["n_train"]} epochs={step["epochs"]} -> retention BP {R:.3f} (puncak {peak:.3f}, akhir {last:.3f})')
            PILOT[bk] = dict(step, retention=round(R, 4), peak=round(peak, 4), last=round(last, 4))
            if R < PILOT_TARGET:
                break
        else:
            PILOT[bk]['warning'] = 'BP tidak kehilangan plastisitas pada semua konfigurasi pilot'
        json.dump(PILOT, open(PILOT_FILE, 'w'), indent=2)
for bk in BENCH:
    BENCH[bk].update(n_train=PILOT[bk]['n_train'], epochs=PILOT[bk]['epochs'])
    BENCH[bk]['n_test'] = min(BENCH[bk]['n_test'], BENCH[bk]['n_train'])
PILOT_OK = all('warning' not in v for v in PILOT.values())
print('PILOT LOLOS — lanjutkan.' if PILOT_OK else 'PERINGATAN: pilot gagal untuk ' + str([k for k, v in PILOT.items() if 'warning' in v]) + '. Hentikan dan laporkan sebelum menjalankan grid.')
print(f'Sisa waktu: {minutes_left():.1f} menit')

## 6. Kalibrasi kecepatan + rencana agar muat ≤ 2 jam
Mengukur detik per tugas untuk tiap benchmark × metode (3 tugas, LR 1e-3), lalu memilih jumlah seed dan tugas. Hasil disimpan ke `plan.json`; jika file itu sudah ada (sesi lanjutan), rencana lama dipakai agar desain eksperimen tetap konsisten.

In [ ]:
PLAN_FILE = f'{RES}/plan.json'
if os.path.exists(PLAN_FILE):
    PLAN = json.load(open(PLAN_FILE))
    print('Rencana lama dipakai:', PLAN)
else:
    spt = {}
    for bk, cfg in BENCH.items():
        data = get_data(bk); spt[bk] = {}
        for m in fp.METHODS:
            assert_drive()
            spt[bk][m] = fp.seconds_per_task(cfg, m, WIDTH, DEPTH, 1e-3, data, DEVICE, CAL, n=CAL_TASKS, **KW)
    tune_s = sum(sum(v.values()) for v in spt.values()) * TUNE_TASKS * len(LR_GRID)
    main_budget = minutes_left() * 60 - tune_s
    PLAN = fp.make_plan(spt, BENCH, main_budget, MIN_TASKS, SEED_OPTIONS)
    PLAN['sec_per_task'] = {b: {m: round(v, 2) for m, v in d.items()} for b, d in spt.items()}
    PLAN['tune_est_min'] = round(tune_s / 60, 1)
    json.dump(PLAN, open(PLAN_FILE, 'w'), indent=2)
for bk in BENCH:
    BENCH[bk]['n_tasks'] = PLAN['n_tasks'][bk]
SEEDS = list(range(PLAN['seeds']))
json.dump(dict(BENCH=BENCH, SEEDS=SEEDS, WIDTH=WIDTH, DEPTH=DEPTH, LR_GRID=LR_GRID, TUNE_TASKS=TUNE_TASKS,
               TUNE_SEED=TUNE_SEED, BUDGET_MIN=BUDGET_MIN, torch=torch.__version__, **KW),
          open(f'{RES}/config.json', 'w'), indent=2)
print(json.dumps(PLAN, indent=2))
print(f'Seed: {SEEDS} | tugas: {PLAN["n_tasks"]} | estimasi utama {PLAN["est_min"]} menit | sisa waktu {minutes_left():.1f} menit')
if 'warning' in PLAN: print('PERINGATAN:', PLAN['warning'])

## 7. Tuning learning rate (sekali saja; hasil disimpan ke `lr_choice.json`)
Protokol: untuk tiap benchmark × metode, grid `LR_GRID` dijalankan pada **5 tugas pertama** dengan seed 100 (tidak dipakai saat evaluasi). LR terpilih = rata-rata akurasi tertinggi. Tuning pada tugas awal saja mencegah pemilihan LR yang 'dioptimalkan' terhadap fase hilangnya plastisitas.

In [ ]:
LR_FILE = f'{RES}/lr_choice.json'
LR = json.load(open(LR_FILE)) if os.path.exists(LR_FILE) else {}

for bk, cfg in BENCH.items():
    LR.setdefault(bk, {})
    data = get_data(bk)
    for m in fp.METHODS:
        if m in LR[bk]:
            continue
        scores = {}
        for lr in LR_GRID:
            tag = f'tune_{bk}_{m}_lr{lr:g}'
            assert_drive()
            fp.run(cfg, m, TUNE_SEED, WIDTH, DEPTH, lr, TUNE, data, DEVICE, ckpt_every=CKPT_EVERY,
                   n_tasks=TUNE_TASKS, tag=tag, log=lambda *a: None, **KW)
            recs = [json.loads(l) for l in open(f'{TUNE}/{tag}.jsonl')]
            scores[lr] = sum(r['acc_end'] for r in recs) / len(recs)
        best = max(scores, key=scores.get)
        LR[bk][m] = best
        json.dump(LR, open(LR_FILE, 'w'), indent=2)       # simpan setiap selesai satu metode
        print(f'{bk:8s} {m:7s} ' + '  '.join(f'{k:g}:{v:.3f}' for k, v in scores.items()) + f'  -> {best:g}')
print(json.dumps(LR, indent=2)); print(f'Sisa waktu: {minutes_left():.1f} menit')

## 8. Eksperimen utama (dijadwalkan agar muat; dapat dilanjutkan otomatis)
Urutan: seed di luar → benchmark → metode. Jadi bila waktu habis, jumlah seed tetap seimbang antar-metode.
Run yang sudah selesai (`.done`) dilewati; run yang terputus dilanjutkan dari checkpoint. Saat batas waktu tercapai, run aktif menyimpan checkpoint dan loop berhenti.

In [ ]:
LR = json.load(open(f'{RES}/lr_choice.json'))
if not PILOT_OK:
    print('PERINGATAN: pilot belum menunjukkan loss of plasticity; hasil grid mungkin tidak menjawab RQ1.')
stop = False
for seed in SEEDS:
    for bk, cfg in BENCH.items():
        data = get_data(bk)
        for m in fp.METHODS:
            if time.time() > DEADLINE:
                stop = True; break
            assert_drive()          # berhenti bila Drive terlepas, bukan menulis ke disk lokal
            try:
                fp.run(cfg, m, seed, WIDTH, DEPTH, LR[bk][m], RES, data, DEVICE,
                       ckpt_every=CKPT_EVERY, deadline=DEADLINE, **KW)
            except Exception as e:      # satu run gagal tidak menghentikan yang lain
                print('ERROR', bk, m, seed, repr(e))
        if stop: break
    if stop:
        print('Batas waktu tercapai: hasil sejauh ini aman di Drive; jalankan ulang notebook untuk melanjutkan.'); break
    print(f'== seed {seed} selesai | sisa {minutes_left():.1f} menit ==')
done = [f for f in os.listdir(RES) if f.endswith('.done')]
print(len(done), 'run selesai dari', len(SEEDS) * len(BENCH) * len(fp.METHODS))

## 8b. Seed tambahan (sampai 5 seed, selama waktu cukup)
Menambah seed di luar rencana awal agar uji Mann–Whitney bisa mencapai p < 0,05 (dengan 3 seed per kelompok, p minimum yang mungkin hanyalah 0,10). Urutannya per seed, jadi jumlah seed tetap seimbang antar-metode bila waktu habis.

In [ ]:
TOTAL_SEEDS = 5
for seed in range(len(SEEDS), TOTAL_SEEDS):
    if time.time() > DEADLINE:
        print('Batas waktu tercapai; seed tambahan dilanjutkan di sesi berikutnya.'); break
    for bk, cfg in BENCH.items():
        data = get_data(bk)
        for m in fp.METHODS:
            if time.time() > DEADLINE: break
            assert_drive()
            try:
                fp.run(cfg, m, seed, WIDTH, DEPTH, LR[bk][m], RES, data, DEVICE,
                       ckpt_every=CKPT_EVERY, deadline=DEADLINE, **KW)
            except Exception as e:
                print('ERROR', bk, m, seed, repr(e))
    print(f'== seed tambahan {seed} | sisa {minutes_left():.1f} menit ==')

## 9. Ablasi lebar jaringan (hanya jika waktu tersisa)
Pada random-label MNIST: lebar 128 dan 1000, metode `bp`, `bpg`, `ff`, 3 seed. LR mengikuti hasil tuning lebar 500. Dilewati otomatis bila sisa waktu < 5 menit; run yang belum selesai bisa dilanjutkan di sesi lain.

In [ ]:
ABL_WIDTHS, ABL_METHODS, ABL_SEEDS = [128, 1000], ['bp', 'bpg', 'ff'], [0, 1, 2]
cfg, data = BENCH['rlmnist'], get_data('rlmnist')
if minutes_left() < 5:
    print(f'Dilewati: sisa waktu {minutes_left():.1f} menit')
else:
    for seed in ABL_SEEDS:
        for w in ABL_WIDTHS:
            for m in ABL_METHODS:
                if time.time() > DEADLINE: break
                assert_drive()
                try:
                    fp.run(cfg, m, seed, w, DEPTH, LR['rlmnist'][m], RES, data, DEVICE,
                           ckpt_every=CKPT_EVERY, deadline=DEADLINE, **KW)
                except Exception as e:
                    print('ERROR', w, m, seed, repr(e))
    print(f'Sisa waktu: {minutes_left():.1f} menit')

## 10. Analisis: tabel, uji statistik, dan gambar
Output di Drive:
- `figures/table_plasticity_summary.csv` — mean ± sd per metode (puncak, akhir, *retention*, unit dorman, *effective rank*)
- `figures/table_per_run.csv`, `figures/table_tests.csv` — Welch t-test, Mann–Whitney U, Cohen's d
- `figures/<bench>_w<width>_<metrik>.pdf/.png` — kurva mean ± sd antar-seed

In [ ]:
import pandas as pd
assert_drive()
df = fp.load_results(RES)
if df.empty:
    raise SystemExit('Belum ada hasil di ' + RES + ' — jalankan sel 8 dulu.')
print(len(df), 'baris;', df['run'].nunique(), 'run')
per_run, summ = fp.plasticity_table(df, window=10, last=10)
per_run.to_csv(f'{FIG}/table_per_run.csv', index=False)
summ.to_csv(f'{FIG}/table_plasticity_summary.csv')
tests = fp.pairwise_tests(per_run, 'retention')
tests_last = fp.pairwise_tests(per_run, 'acc_last')
pd.concat([tests, tests_last]).to_csv(f'{FIG}/table_tests.csv', index=False)

metrics = {'acc_end': 'Accuracy on task data (end of task)', 'acc_q1': 'Accuracy (25% of task)',
           'dormant_mean': 'Dormant units (fraction, true-label probe)', 'erank_mean': 'Effective rank (true-label probe)',
           'dormant_neutral_mean': 'Dormant units (neutral-label probe)', 'erank_neutral_mean': 'Effective rank (neutral-label probe)',
           'wmag_mean': 'Mean |weight|', 'goodness_sep': 'Goodness separation'}
for (bench, width), _ in df.groupby(['bench', 'width']):
    for k, lab in metrics.items():
        for ext in ('pdf', 'png'):
            fp.plot_curves(df, bench, width, k, f'{FIG}/{bench}_w{width}_{k}.{ext}', ylabel=lab)
print('Gambar & tabel tersimpan di', FIG)
pd.set_option('display.width', 200)
cols = ['acc_peak', 'acc_last', 'retention', 'dormant_last', 'dormant_neutral_last', 'erank_last', 'erank_neutral_last']
display(summ[cols].round(3))
display(tests.round(4))

# rezim kontrol (v1): dianalisis ulang dengan tabel yang sama, disimpan dengan awalan control_
if os.path.isdir(CONTROL_RES) and any(f.endswith('.jsonl') for f in os.listdir(CONTROL_RES)):
    dfc = fp.load_results(CONTROL_RES)
    per_c, summ_c = fp.plasticity_table(dfc, window=10, last=10)
    per_c.to_csv(f'{FIG}/control_table_per_run.csv', index=False)
    summ_c.to_csv(f'{FIG}/control_table_plasticity_summary.csv')
    pd.concat([fp.pairwise_tests(per_c, 'retention'), fp.pairwise_tests(per_c, 'acc_last')]).to_csv(
        f'{FIG}/control_table_tests.csv', index=False)
    print('Rezim kontrol (v1; probe label netral untuk FF/BP-G):')
    display(summ_c[['acc_peak', 'acc_last', 'retention', 'dormant_last', 'erank_last']].round(3))

## 10c. Gambar naskah (Fig. 1–5, siap kirim ke jurnal)
Membuat lima gambar sesuai rencana naskah, masing-masing sebagai PDF vektor + PNG 300 dpi, lebar 90 mm (satu kolom) atau 190 mm (lebar penuh), palet ramah buta warna. Disimpan di `figures_v2/paper/` dan ikut masuk ZIP.

In [ ]:
%%writefile /content/drive/MyDrive/ff_plasticity/paper_figures.py
# paper_figures.py — publication figures for the FF vs BP plasticity manuscript.
# Elsevier: vector PDF + 300-dpi PNG/TIFF, one file per figure, colour-blind-safe palette,
# single column = 90 mm, full width = 190 mm, text >= 7 pt.
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

MM = 1 / 25.4
SINGLE, FULL = 90 * MM, 190 * MM
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7, "axes.linewidth": 0.6,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6, "lines.linewidth": 1.3,
    "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42, "ps.fonttype": 42,
    "savefig.dpi": 300,
})
# validated (CVD-safe) categorical palette: BP, BP-N, BP-G, FF ; reset = same hue, dashed
COL = {"bp": "#0072B2", "bp_cbp": "#0072B2", "bpn": "#009E73", "bpg": "#7B5EA7",
       "ff": "#D55E00", "ff_cbp": "#D55E00"}
LS = {"bp": "-", "bp_cbp": "--", "bpn": "-", "bpg": "-", "ff": "-", "ff_cbp": "--"}
MK = {"bp": "o", "bp_cbp": "P", "bpn": "s", "bpg": "D", "ff": "^", "ff_cbp": "v"}
NAME = {"bp": "BP", "bp_cbp": "BP + reset", "bpn": "BP-N", "bpg": "BP-G", "ff": "FF", "ff_cbp": "FF + reset"}
ORDER = ["bp", "bp_cbp", "bpn", "bpg", "ff", "ff_cbp"]
STREAM = {"rlmnist": "Random-label MNIST", "rlcifar": "Random-label CIFAR-10"}


def _save(fig, path_noext):
    for ext in ("pdf", "png"):
        fig.savefig(f"{path_noext}.{ext}", bbox_inches="tight", pad_inches=0.02)
    plt.close(fig)
    return path_noext


def _curve(ax, df, method, metric, smooth=3):
    g = df[df.method == method]
    if g.empty:
        return False
    piv = g.pivot_table(index="task", columns="seed", values=metric).rolling(smooth, min_periods=1).mean()
    mu, sd = piv.mean(1), piv.std(1).fillna(0)
    ax.plot(mu.index, mu.values, color=COL[method], ls=LS[method], label=NAME[method])
    ax.fill_between(mu.index, mu - sd, mu + sd, color=COL[method], alpha=0.15, lw=0)
    return True


# ---------------------------------------------------------------- Fig. 1
def fig1_conditions(path):
    """Schematic of the conditions: normalisation, objective and gradient scope."""
    fig, axes = plt.subplots(1, 4, figsize=(FULL, 62 * MM))
    specs = [("BP", False, "softmax", "global"), ("BP-N", True, "softmax", "global"),
             ("BP-G", True, "goodness", "global"), ("FF", True, "goodness", "local")]
    grad = "#C0392B"
    for ax, (title, norm, obj, scope) in zip(axes, specs):
        ax.set_xlim(-1, 11); ax.set_ylim(0, 13); ax.axis("off")
        ax.set_title(title, fontweight="bold", pad=2)
        ys = [2.2, 5.0, 7.8]
        ax.add_patch(FancyBboxPatch((1.7, 0.2), 5.6, 1.1, boxstyle="round,pad=0.05,rounding_size=0.3",
                                    fc="white", ec="0.35", lw=0.7))
        ax.text(4.5, 0.75, "image + label" if obj == "goodness" else "image", ha="center", va="center", fontsize=6.3)
        prev_top = 1.3
        for i, y in enumerate(ys):
            if norm:
                ax.text(4.75, y - 0.62, "RMS norm", ha="left", va="center", fontsize=5.6, color="0.3")
            ax.annotate("", xy=(4.5, y - 0.05), xytext=(4.5, prev_top),
                        arrowprops=dict(arrowstyle="-|>", lw=0.6, color="0.35", mutation_scale=6))
            ax.add_patch(FancyBboxPatch((1.7, y), 5.6, 1.3, boxstyle="round,pad=0.05,rounding_size=0.3",
                                        fc="#EEF3F8", ec="0.35", lw=0.7))
            ax.text(4.5, y + 0.65, f"layer {i + 1}", ha="center", va="center", fontsize=6.5)
            if obj == "goodness":
                ax.text(7.6, y + 0.65, rf"$\mathcal{{L}}_{i + 1}$", ha="left", va="center", fontsize=7, color=grad)
            prev_top = y + 1.3
        if obj == "softmax":
            ax.annotate("", xy=(4.5, 10.35), xytext=(4.5, prev_top),
                        arrowprops=dict(arrowstyle="-|>", lw=0.6, color="0.35", mutation_scale=6))
            ax.add_patch(FancyBboxPatch((1.7, 10.4), 5.6, 1.2, boxstyle="round,pad=0.05,rounding_size=0.3",
                                        fc="white", ec="0.35", lw=0.7))
            ax.text(4.5, 11.0, "softmax · CE", ha="center", va="center", fontsize=6.3)
            ax.text(7.6, 11.0, r"$\mathcal{L}$", ha="left", va="center", fontsize=7, color=grad)
        else:
            ax.text(4.5, 10.7, r"loss $=\sum_l \mathcal{L}_l$", ha="center", va="center", fontsize=6.5)
        if scope == "global":
            top = 11.0 if obj == "softmax" else 8.45
            ax.add_patch(FancyArrowPatch((0.9, top), (0.9, 2.6), arrowstyle="-|>", mutation_scale=7,
                                         color=grad, lw=1.1, connectionstyle="arc3,rad=0"))
            ax.text(0.3, (top + 2.6) / 2, "global gradient", rotation=90, ha="center", va="center",
                    fontsize=6, color=grad)
        else:
            for y in ys:
                ax.add_patch(FancyArrowPatch((0.9, y + 1.2), (0.9, y + 0.1), arrowstyle="-|>", mutation_scale=6,
                                             color=grad, lw=1.1))
            ax.text(-0.2, 5.6, "local gradient per layer", rotation=90, ha="center",
                    va="center", fontsize=6, color=grad)
            ax.text(4.5, 10.0, "stop-gradient between layers", ha="center", va="center", fontsize=5.6, color=grad)
    fig.subplots_adjust(wspace=0.05, left=0.01, right=0.99, top=0.9, bottom=0.02)
    return _save(fig, path)


# ---------------------------------------------------------------- Fig. 2
def fig2_accuracy(df, path, width=500):
    fig, axes = plt.subplots(1, 2, figsize=(FULL, 62 * MM), sharey=True)
    for ax, (bk, title) in zip(axes, STREAM.items()):
        sub = df[(df.bench == bk) & (df.width == width)]
        for m in ORDER:
            _curve(ax, sub, m, "acc_end")
        ax.axhline(0.1, color="0.5", lw=0.6, ls=":")
        ax.text(sub.task.max() if len(sub) else 80, 0.115, "chance", ha="right", va="bottom", fontsize=6, color="0.4")
        ax.set_title(title); ax.set_xlabel("Task number"); ax.set_ylim(0, 1.04)
        ax.grid(alpha=0.25, lw=0.4)
    axes[0].set_ylabel("Accuracy at end of task")
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 1.06))
    fig.tight_layout(w_pad=1.2)
    return _save(fig, path)


# ---------------------------------------------------------------- Fig. 3
def fig3_mechanisms(df, path, width=500):
    fig, axes = plt.subplots(2, 2, figsize=(FULL, 108 * MM), sharex=True)
    rows = [("dormant_mean", "Dormant units (fraction)"), ("erank_mean", "Effective rank")]
    for r, (metric, lab) in enumerate(rows):
        for c, (bk, title) in enumerate(STREAM.items()):
            ax = axes[r, c]
            sub = df[(df.bench == bk) & (df.width == width)]
            for m in ORDER:
                _curve(ax, sub, m, metric)
            if r == 0:
                ax.set_title(title); ax.set_ylim(0, 1)
            else:
                ax.set_xlabel("Task number"); ax.set_ylim(bottom=0)
            if c == 0:
                ax.set_ylabel(lab)
            ax.grid(alpha=0.25, lw=0.4)
            ax.text(-0.13 if c == 0 else -0.06, 1.04, "abcd"[2 * r + c], transform=ax.transAxes,
                    fontweight="bold", fontsize=8.5)
    h, l = axes[0, 0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper center", ncol=6, frameon=False, bbox_to_anchor=(0.5, 1.03))
    fig.tight_layout(h_pad=0.8, w_pad=1.2)
    return _save(fig, path)


# ---------------------------------------------------------------- Fig. 4
def fig4_rank_vs_retention(points, path, rho=None):
    """points: list of dicts {method, bench, width, R, erank}."""
    fig, ax = plt.subplots(figsize=(SINGLE, 80 * MM))
    seen = set()
    for p in points:
        main = p["width"] == 500
        filled = p["bench"] == "rlmnist"
        lab = NAME[p["method"]] if (main and p["method"] not in seen) else None
        if main:
            seen.add(p["method"])
        ax.scatter(p["erank"], p["R"], s=26 if main else 14, marker=MK[p["method"]],
                   facecolors=COL[p["method"]] if filled else "white", edgecolors=COL[p["method"]],
                   linewidths=0.9, alpha=1 if main else 0.6, label=lab, zorder=3)
    ax.set_xscale("log"); ax.set_xlabel("Effective rank (last 10 tasks, log scale)")
    ax.set_ylabel("Retention R"); ax.set_ylim(0, 1.05); ax.grid(alpha=0.25, lw=0.4, which="major")
    leg = ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), frameon=False, ncol=3, handletextpad=0.3, columnspacing=1.0)
    for hnd in leg.legend_handles:
        hnd.set_facecolor(hnd.get_edgecolor())
    if rho is not None:
        ax.text(0.03, 0.96, rho, transform=ax.transAxes, va="top", fontsize=7)
    ax.text(0.03, 0.80, "filled: MNIST · open: CIFAR-10\nsmall: width 128 / 1,000",
            transform=ax.transAxes, va="top", fontsize=6, color="0.35")
    fig.tight_layout()
    return _save(fig, path)


# ---------------------------------------------------------------- Fig. 5
def fig5_width(rows, path):
    """rows: list of dicts {method, width, R_mean, R_sd} (random-label MNIST)."""
    fig, ax = plt.subplots(figsize=(SINGLE, 62 * MM))
    for m in ("bp", "bpg", "ff"):
        rr = sorted([r for r in rows if r["method"] == m], key=lambda r: r["width"])
        x = [r["width"] for r in rr]; y = [r["R_mean"] for r in rr]; e = [r["R_sd"] for r in rr]
        ax.errorbar(x, y, yerr=e, color=COL[m], marker=MK[m], ms=4.5, capsize=2, lw=1.2, label=NAME[m])
        ax.text(x[-1] * 1.08, y[-1], NAME[m], color="0.15", fontsize=6.5, va="center")
    ax.set_xscale("log"); ax.set_xticks([128, 500, 1000]); ax.set_xticklabels(["128", "500", "1,000"])
    ax.set_xlim(100, 1600); ax.set_ylim(0, 1.05)
    ax.set_xlabel("Hidden units per layer"); ax.set_ylabel("Retention R")
    ax.grid(alpha=0.25, lw=0.4); ax.legend(loc="center left", frameon=False)
    fig.tight_layout()
    return _save(fig, path)


def summary_points(per_run):
    """Aggregate plasticity_table per_run output into points for fig4/fig5."""
    g = per_run.groupby(["bench", "method", "width"]).agg(R=("retention", "mean"), R_sd=("retention", "std"),
                                                           erank=("erank_last", "mean")).reset_index()
    return g


In [ ]:
import importlib, paper_figures as pfig
importlib.reload(pfig)
PAPER = f'{FIG}/paper'; os.makedirs(PAPER, exist_ok=True)
assert_drive()
pfig.fig1_conditions(f'{PAPER}/Figure_1')
pfig.fig2_accuracy(df, f'{PAPER}/Figure_2', width=WIDTH)
pfig.fig3_mechanisms(df, f'{PAPER}/Figure_3', width=WIDTH)
agg = pfig.summary_points(per_run)
pts = [dict(bench=r.bench, method=r.method, width=r.width, R=r.R, erank=r.erank) for r in agg.itertuples()]
from scipy.stats import spearmanr
m500 = agg[agg.width == WIDTH]
rho = f'Spearman ρ = {spearmanr(m500.R, m500.erank)[0]:.2f} (width {WIDTH}, n = {len(m500)})\nρ = {spearmanr(agg.R, agg.erank)[0]:.2f} (all widths, n = {len(agg)})'
pfig.fig4_rank_vs_retention(pts, f'{PAPER}/Figure_4', rho=rho)
wr = agg[(agg.bench == 'rlmnist') & agg.method.isin(['bp', 'bpg', 'ff'])]
pfig.fig5_width([dict(method=r.method, width=r.width, R_mean=r.R, R_sd=r.R_sd) for r in wr.itertuples()], f'{PAPER}/Figure_5')
print(rho.replace(chr(10), ' | ')); print('Gambar naskah:', sorted(os.listdir(PAPER)))

## 10b. Paket hasil sebagai ZIP (disimpan di Drive + diunduh otomatis)
Membuat satu file ZIP berisi semua yang dibutuhkan untuk naskah dan untuk dikirim/diarsipkan: log per tugas (`*.jsonl`), konfigurasi (`config.json`, `pilot.json`, `plan.json`, `lr_choice.json`), tabel CSV + gambar PDF/PNG (v2/v3 dan kontrol v1), serta `ffplast.py`. Data cache dan checkpoint **tidak** ikut (besar dan tidak diperlukan). ZIP disimpan di `MyDrive/ff_plasticity/exports/` lalu diunduh ke komputer Anda (jika browser meminta izin unduhan ganda, pilih *Allow*).

In [ ]:
import zipfile, datetime
EXP = f'{BASE}/exports'; os.makedirs(EXP, exist_ok=True)
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
ZIP_PATH = f'{EXP}/ff_plasticity_results_{stamp}.zip'
KEEP_EXT = ('.jsonl', '.json', '.csv', '.pdf', '.png', '.py', '.done')
SOURCES = [RES, FIG, CONTROL_RES, f'{BASE}/figures']        # v2/v3 + kontrol v1
n_files = 0
assert_drive()
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for src in SOURCES:
        if not os.path.isdir(src):
            continue
        for dp, _, fs in os.walk(src):
            for f in fs:
                if f.endswith(KEEP_EXT):
                    full = os.path.join(dp, f)
                    z.write(full, os.path.relpath(full, BASE)); n_files += 1
    z.write(f'{BASE}/ffplast.py', 'ffplast.py'); n_files += 1
    z.writestr('README.txt', f"""FF vs BP loss-of-plasticity results — exported {stamp}
results_v2/   : per-task logs (*.jsonl), pilot.json, plan.json, lr_choice.json, config.json (main regime)
figures_v2/   : table_plasticity_summary.csv, table_tests.csv, table_per_run.csv, control_*.csv, curves (PDF/PNG)
results/      : control regime (v1) logs;  figures/ : v1 figures
ffplast.py    : experiment library used for every run
""")
os.sync()
print(f'ZIP: {ZIP_PATH} | {n_files} file | {os.path.getsize(ZIP_PATH) / 1e6:.1f} MB')
try:
    from google.colab import files
    files.download(ZIP_PATH)                                 # unduh ke komputer
except Exception as e:
    print('Unduhan otomatis tidak tersedia; ambil ZIP dari Google Drive. (' + repr(e) + ')')

## 11. Sinkronkan ke Google Drive (jalankan di akhir sesi)
Setiap hasil sudah ditulis langsung ke Drive selama eksperimen: metrik per tugas (`*.jsonl`, setiap tugas), checkpoint (`*.ckpt`, tiap 10 tugas dan saat waktu habis), serta penanda selesai (`*.done`). Colab mengunggah file ke cloud di latar belakang; sel ini memaksa semua unggahan tertunda selesai sebelum Anda menutup sesi, lalu menampilkan isi folder untuk dicek.

In [ ]:
os.sync()
n = {k: len([f for f in os.listdir(RES) if f.endswith(k)]) for k in ('.jsonl', '.ckpt', '.done')}
size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(BASE) for f in fs) / 1e6
print(f'Di Drive: {n[".jsonl"]} log run, {n[".done"]} run selesai, {n[".ckpt"]} checkpoint run belum selesai | total {size:.0f} MB')
drive.flush_and_unmount()
print('Semua file sudah tersinkron ke Google Drive. Cek di drive.google.com → MyDrive/ff_plasticity.')
print('Catatan: Drive kini dilepas; untuk lanjut bekerja, jalankan ulang sel 1.')

## 12. Catatan untuk naskah
- Analisis memakai run yang sudah ada (termasuk yang belum selesai); laporkan hanya run lengkap (`n_tasks` di tabel = jumlah tugas yang ditempuh).
- Jika BP tidak menunjukkan penurunan (retention ≈ 1), plastisitas belum hilang dalam jumlah tugas ini: tambah tugas di sesi berikutnya (hapus `plan.json`, naikkan `BUDGET_MIN`).
- Laporkan: arsitektur (3×500, ReLU, Kaiming), optimizer & LR terpilih (`lr_choice.json`), jumlah tugas, ukuran data per tugas, batch, epoch, 5 seed, dan `config.json`.
- Data availability (wajib di *Neural Networks*, Option C): unggah `ffplast.py`, notebook ini, `config.json`, `pilot.json`, `plan.json`, `lr_choice.json`, dan semua `*.jsonl` ke GitHub + Zenodo (DOI), lalu sitasi di naskah.
- `bpg` adalah kontrol penting: tanpa itu reviewer bisa berargumen bahwa perbedaan FF vs BP berasal dari objektif *goodness*/normalisasi, bukan dari lokalitas.